In [1]:
import pandas as pd
import numpy as np

ufos = pd.read_csv('./data/ufos.csv')
ufos.head()

,datetime,city,state,country,shape,duration (seconds),duration (hours/min),comments,date posted,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700.0,45 minutes,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.384210,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.200000,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833
4,10/10/1960 20:00,kaneohe,hi,us,light,900.0,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611


In [2]:
ufos = pd.DataFrame({'Seconds': ufos['duration (seconds)'], 'Country': ufos['country'],'Latitude': ufos['latitude'],'Longitude': ufos['longitude']})

ufos.Country.unique()

<StringArray>
['us', nan, 'gb', 'ca', 'au', 'de']
Length: 6, dtype: str

In [3]:
ufos.head()

,Seconds,Country,Latitude,Longitude
0,2700.0,us,29.883056,-97.941111
1,7200.0,NaN,29.384210,-98.581082
2,20.0,gb,53.200000,-2.916667
3,20.0,us,28.978333,-96.645833
4,900.0,us,21.418056,-157.803611


In [4]:
print(ufos.shape)

(80332, 4)


In [5]:
ufos.isna().sum()

Seconds         0
Country      9670
Latitude        0
Longitude       0
dtype: int64

I think I can find the countries that are realted to latitude and longitude
okay I find the api so lets fill the blanks

In [6]:
import requests
import time
def find_country_code(latitude: float, longitude: float) -> str | None:
    base_url = "https://api.bigdatacloud.net/data/reverse-geocode-client"
    time.sleep(1)
    parameters = {
        "latitude": latitude,
        "longitude": longitude,
        "localityLanguage": "en"
    }
    response = requests.get(base_url, params=parameters)
    if response.status_code == 200:
        data = response.json()
        return data.get("countryCode").lower()
    # since data in the dataset is lower 
    else:
        return None

In [7]:
null_indexes = ufos['Country'].isna()
ufos.loc[null_indexes, 'Country'] = ufos.loc[null_indexes].apply(lambda row: find_country_code(row['Latitude'], row['Longitude']), axis=1)

SSLError: HTTPSConnectionPool(host='api.bigdatacloud.net', port=443): Max retries exceeded with url: /data/reverse-geocode-client?latitude=29.369722&longitude=47.978333&localityLanguage=en (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1010)')))

In [ ]:
ufos['Country'].isna().sum()

np.int64(9670)

since the api doesnt allow me to repair data I will use the existing ones 

In [9]:
ufos.isnull().sum()

Seconds         0
Country      9670
Latitude        0
Longitude       0
dtype: int64

In [10]:
ufos.dropna(inplace=True)

In [11]:
ufos.isnull().sum()

Seconds      0
Country      0
Latitude     0
Longitude    0
dtype: int64

In [12]:
ufos.shape

(70662, 4)

In [13]:
ufos.head()

,Seconds,Country,Latitude,Longitude
0,2700.0,us,29.883056,-97.941111
2,20.0,gb,53.200000,-2.916667
3,20.0,us,28.978333,-96.645833
4,900.0,us,21.418056,-157.803611
5,300.0,us,36.595000,-82.188889


In [14]:
ufos = ufos[(ufos['Seconds'] >= 1) & (ufos['Seconds'] <= 60)]

ufos.info()

<class 'pandas.DataFrame'>
Index: 25863 entries, 2 to 80330
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Seconds    25863 non-null  float64
 1   Country    25863 non-null  str    
 2   Latitude   25863 non-null  float64
 3   Longitude  25863 non-null  float64
dtypes: float64(3), str(1)
memory usage: 1010.3 KB


In [15]:
from sklearn.preprocessing import LabelEncoder


ufos['Country'] = LabelEncoder().fit_transform(ufos['Country'])


ufos.head()

,Seconds,Country,Latitude,Longitude
2,20.0,3,53.200000,-2.916667
3,20.0,4,28.978333,-96.645833
14,30.0,4,35.823889,-80.253611
23,60.0,4,45.582778,-122.352222
24,3.0,3,51.783333,-0.783333


In [ ]:

ufos['Country'].unique()
# we only have 5 countries

array([3, 4, 1, 0, 2])

In [17]:
    from sklearn.model_selection import train_test_split
    
    Selected_features = ['Seconds','Latitude','Longitude']
    
    X = ufos[Selected_features]
    y = ufos['Country']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [22]:
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print(classification_report(y_test, predictions))
print('Predicted labels: ', predictions)
print('Accuracy: ', accuracy_score(y_test, predictions))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        41
           1       0.85      0.46      0.60       250
           2       1.00      1.00      1.00         8
           3       1.00      1.00      1.00       131
           4       0.97      1.00      0.98      4743

    accuracy                           0.97      5173
   macro avg       0.96      0.89      0.92      5173
weighted avg       0.97      0.97      0.97      5173

Predicted labels:  [4 4 4 ... 3 4 4]
Accuracy:  0.970036729170694


now we will create the open model for other people to use   

In [26]:
import pickle
model_filename = 'ufo-model.pkl'
pickle.dump(model, open(model_filename,'wb'))

loaded_model = pickle.load(open('ufo-model.pkl','rb'))
input_data = input_data = pd.DataFrame([[50, 44, -12]], columns=['Seconds','Latitude','Longitude'])
print(loaded_model.predict(input_data))

[3]
